<a href="https://colab.research.google.com/github/Tonys-Coding/Netflix-Rating-Predictor/blob/main/My_Netflix_Prediction_Program.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.feature_extraction.text import TfidfVectorizer
import scipy.sparse as sp
#from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor


In [2]:

import kagglehub
import os

#Download the dataset
##########################
# Paste code from Kaggle
##########################

# Download latest version
path = kagglehub.dataset_download("bhargavchirumamilla/netflix-movies-and-tv-shows-till-2025")

print("Path to dataset files:", path)

#See what files are in the downloaded folder
files = os.listdir(path)
print("Files:", files)

#Load the CSV into a DataFrame
csv_file = [f for f in files if f.endswith('.csv')][0]
##########################
df = pd.read_csv(os.path.join(path, csv_file))
##########################

#Explore the data
print(df.shape)        # rows and columns
print(df.head())       # first 5 rows
print(df.dtypes)       # column types
print(df.isnull().sum()) # missing values



100%|██████████| 6.17M/6.17M [00:00<00:00, 24.1MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/bhargavchirumamilla/netflix-movies-and-tv-shows-till-2025/versions/1
Files: ['netflix_movies_detailed_up_to_2025.csv', 'netflix_tv_shows_detailed_up_to_2025.csv']
(16000, 18)
   show_id   type                                         title  \
0    10192  Movie                           Shrek Forever After   
1    27205  Movie                                     Inception   
2    12444  Movie  Harry Potter and the Deathly Hallows: Part 1   
3    38757  Movie                                       Tangled   
4    10191  Movie                      How to Train Your Dragon   

                      director  \
0                Mike Mitchell   
1            Christopher Nolan   
2                  David Yates   
3   Byron Howard, Nathan Greno   
4  Chris Sanders, Dean DeBlois   

                                                cast  \
0  Mike Myers, Eddie Murphy, Cameron Diaz, Antoni...   
1  Leonardo DiCaprio, Joseph Gordon-Levitt, Ken W.

In [3]:
# Add features to data frame
##########################
df = df.dropna(subset=['rating', 'release_year', 'genres', 'description'])
##########################

# Encode type
df['is_tv'] = (df['type'].astype(str) == 'TV Show').astype(int)

# Extract date features
df['date_added'] = pd.to_datetime(df['date_added'], errors='coerce')
df['added_year'] = df['date_added'].dt.year.fillna(0).astype(int)
df['added_month'] = df['date_added'].dt.month.fillna(0).astype(int)


In [4]:
#Feature engineering

# Top 10 genres
top_genres = df['genres'].str.split(', ').explode().value_counts().head(10).index
for genre in top_genres:
    df[f'genre_{genre}'] = df['genres'].str.contains(genre, na=False).astype(int)

# Top 10 languages
top_langs = df['language'].dropna().value_counts().head(10).index
for lang in top_langs:
    df[f'lang_{lang}'] = (df['language'] == lang).astype(int)

# Top 20 actors
top_actors = df['cast'].dropna().str.split(', ').explode().value_counts().head(20).index
for actor in top_actors:
    df[f'actor_{actor}'] = df['cast'].str.contains(actor, na=False).astype(int)

# Top 20 directors
top_directors = df['director'].dropna().value_counts().head(20).index
for director in top_directors:
    df[f'director_{director}'] = df['director'].str.contains(director, na=False).astype(int)

# Top 15 countries
top_countries = df['country'].dropna().value_counts().head(15).index
for country in top_countries:
    df[f'country_{country}'] = df['country'].str.contains(country, na=False).astype(int)

In [5]:
feature_cols = [
    'is_tv', 'release_year', 'popularity', 'vote_count',
    'added_year', 'added_month'
] + [f'genre_{g}' for g in top_genres] \
  + [f'lang_{l}' for l in top_langs] \
  + [f'actor_{a}' for a in top_actors] \
  + [f'director_{d}' for d in top_directors] \
  + [f'country_{c}' for c in top_countries]

##########################
X = df[feature_cols]
y = df['rating']
##########################

In [6]:
tfidf = TfidfVectorizer(max_features=100, stop_words='english')
desc_features = tfidf.fit_transform(df['description'].fillna(''))

# Combine structured features with TF-IDF features

##########################
X_combined = sp.hstack([sp.csr_matrix(X.values), desc_features])
##########################

In [8]:
#split data into training and testing data
##########################
X_train, X_test, y_train, y_test = train_test_split(
    X_combined, y, test_size=0.2, random_state=42
 )
##########################

#train Random Forest model
##########################
model = XGBRegressor(n_estimators=200, learning_rate=0.05, random_state=42)
##########################
model.fit(X_train, y_train)

#evaluate model
y_pred = model.predict(X_test)
print(f"MAE: {mean_absolute_error(y_test, y_pred):.3f}")
print(f"R²:  {r2_score(y_test, y_pred):.3f}")

MAE: 0.560
R²:  0.764
